# Pandas Advanced Practice for Data Science

A one-hour hands-on tour of the *most useful* advanced Pandas skills:

1. **Group & Aggregate** — answer "per group" questions
2. **Concatenate & Merge** — combine several tables
3. **Remove duplicates & bin values** — clean data
4. **Pivot & Crosstab** — reshape data
5. **Visualize** — turn tables into charts
6. **Time-Series** — work with dates, resampling, and rolling windows

- **Duration:** ~1 hour
- **Platform:** Google Colab (no setup needed)
- **How to use it:** Type each cell along with the instructor. Press `Shift + Enter` to run a cell.

## 1. Setup

We use `pandas` for tables, `numpy` for numbers, `matplotlib` for plots, and `seaborn` for its built-in practice datasets (Titanic, flights, tips).

In [ ]:
# Import pandas (data tables) with the standard nickname 'pd'
import pandas as pd
# Import numpy (numbers) with the standard nickname 'np'
import numpy as np
# Import matplotlib for plotting, with the standard nickname 'plt'
import matplotlib.pyplot as plt
# Import seaborn — includes built-in datasets and nicer default plot styling
import seaborn as sns
# Turn on a clean dark-grid look for all our plots
sns.set_style("darkgrid")

In [ ]:
# Load the built-in Titanic dataset — each row is one passenger
titanic_data = sns.load_dataset('titanic')
# Preview the first 5 rows
titanic_data.head()

## 2. Group & Aggregate with `groupby`

Question: *"What was the average fare in each passenger class?"* — that's a **per-group** question. We use `groupby` to answer it.

In [ ]:
# Group the Titanic dataset by the 'class' column (First, Second, Third)
# observed=False keeps the current behavior and silences a warning
by_class = titanic_data.groupby("class", observed=False)
# .size() tells us how many passengers are in each group
by_class.size()

In [ ]:
# Get the maximum age from each class
# Shorthand: after groupby, we can pick a column with .age or ['age']
by_class.age.max()

In [ ]:
# Run SEVERAL statistics at once on the 'fare' column using .agg([...])
by_class.fare.agg(['max', 'min', 'count', 'median', 'mean'])

In [ ]:
# We can group by MORE than one column — here: class AND sex
# The result has one row per (class, sex) combination
titanic_data.groupby(["class", "sex"], observed=False)["fare"].mean()

## 3. Concatenate & Merge

- **Concatenate** = stack tables on top of (or beside) each other.
- **Merge** = combine two tables by matching values in a shared column (like SQL JOIN).

In [ ]:
# Filter passengers into two smaller DataFrames — one per class
first_class  = titanic_data[titanic_data["class"] == "First"]
second_class = titanic_data[titanic_data["class"] == "Second"]
# Show the shape (rows, columns) of each — First has 216, Second has 184
print("first:",  first_class.shape)
print("second:", second_class.shape)

In [ ]:
# pd.concat stacks the tables on top of each other (216 + 184 = 400 rows)
# ignore_index=True renumbers rows so we don't have duplicate labels
combined = pd.concat([first_class, second_class], ignore_index=True)
print(combined.shape)

In [ ]:
# Build two small demo DataFrames to practice merging
# Both share the 'Subject' column — that's the KEY we'll match on
scores1_df = pd.DataFrame([
    {'Subject': 'Mathematics', 'Score': 85},
    {'Subject': 'History',     'Score': 98},
    {'Subject': 'English',     'Score': 76},
    {'Subject': 'Chemistry',   'Score': 72},
])
scores2_df = pd.DataFrame([
    {'Subject': 'Arts',        'Grade': 'C'},
    {'Subject': 'Physics',     'Grade': 'C'},
    {'Subject': 'English',     'Grade': 'A'},
    {'Subject': 'Chemistry',   'Grade': 'A'},
])
print(scores1_df)
print(scores2_df)

In [ ]:
# INNER merge — keep ONLY subjects that exist in BOTH tables
# Result: English + Chemistry (the two shared subjects)
scores1_df.merge(scores2_df, on='Subject', how='inner')

In [ ]:
# LEFT merge — keep ALL rows from the LEFT table, fill missing right-side values with NaN
scores1_df.merge(scores2_df, on='Subject', how='left')

In [ ]:
# OUTER merge — keep ALL rows from BOTH tables; missing values become NaN
scores1_df.merge(scores2_df, on='Subject', how='outer')

## 4. Removing Duplicates and Binning Values

In [ ]:
# Small demo DataFrame with two identical rows
scores = pd.DataFrame({
    'Subject': ['Math', 'English', 'History', 'History', 'Arts'],
    'Score':   [85, 91, 95, 95, 88],
})
scores

In [ ]:
# drop_duplicates() removes exact duplicate rows (keeping the first by default)
scores.drop_duplicates()

In [ ]:
# We can also check duplicates only on specific columns
# subset=['Score'] means: treat rows as duplicates if their 'Score' is the same
scores.drop_duplicates(subset=['Score'])

In [ ]:
# BINNING = turn continuous numbers into categories
# Here we group ages into 4 buckets: toddler, young, adult, senior
titanic_data['age_group'] = pd.cut(
    x=titanic_data['age'],                          # the number column to bin
    bins=[0, 5, 20, 60, 100],                        # the bucket edges
    labels=["toddler", "young", "adult", "senior"]   # names for each bucket
)
# Show how many passengers ended up in each age group
titanic_data['age_group'].value_counts()

## 5. Pivot Table & Crosstab

- **pivot_table** reshapes long data into a wide summary table.
- **crosstab** counts how often each combination of two categories appears.

In [ ]:
# Load the built-in 'flights' dataset — passengers per month from 1949 to 1960
flights_data = sns.load_dataset('flights')
flights_data.head()

In [ ]:
# Reshape flights into a grid: months as rows, years as columns, passengers as values
flights_pivot = flights_data.pivot_table(
    index='month',        # rows
    columns='year',       # columns
    values='passengers',  # what to put inside each cell
    observed=False
)
flights_pivot.head()

In [ ]:
# crosstab counts how many passengers of each SEX travelled in each CLASS
# margins=True adds a total 'All' row and column
pd.crosstab(titanic_data["class"], titanic_data["sex"], margins=True)

## 6. Visualisation with Pandas

Pandas has plotting built right in. Under the hood it uses Matplotlib, but the code is much shorter.

In [ ]:
# HISTOGRAM — shows the distribution of a numeric column
# bins=20 splits the data into 20 groups; color= sets the bar color
titanic_data['age'].hist(bins=20, color='orange')
# plt.show() forces the plot to display right now
plt.show()

In [ ]:
# LINE plot — great for values that change over time
# figsize sets the width and height of the chart (in inches)
flights_data.plot.line(y='passengers', figsize=(8, 4))
plt.show()

In [ ]:
# SCATTER plot — one dot per row; shows the relationship between two numeric columns
flights_data.plot.scatter(x='year', y='passengers', color='red', figsize=(8, 4))
plt.show()

In [ ]:
# BAR chart — compare a number across a few categories
# Step 1: compute the average age for each sex
sex_mean = titanic_data.groupby("sex", observed=False)["age"].mean()
# Step 2: plot the resulting Series directly as a bar chart
sex_mean.plot(kind='bar', color='steelblue', title='Average Age by Sex', figsize=(6, 4))
plt.show()

In [ ]:
# BOX plot — shows the spread (min, quartiles, max) of numeric columns
# 'age' and 'fare' are on very different scales, but the box plot still works
titanic_data[['age', 'fare']].plot.box(figsize=(6, 4))
plt.show()

In [ ]:
# PIE chart — shows how a whole is split into parts
# autopct puts the percentage on each slice
titanic_data["class"].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',   # format the percentage label
    figsize=(6, 6),
    title='Passengers by Class'
)
plt.ylabel('')          # hide the extra 'class' label on the y-axis
plt.show()

In [ ]:
# KDE plot (Kernel Density Estimate) — a smooth version of a histogram
# We use the built-in 'tips' dataset (restaurant tips)
tips_data = sns.load_dataset('tips')
tips_data['tip'].plot.kde(figsize=(8, 4), color='green', title='Distribution of Tips')
plt.show()

## 7. Time-Series Data

A **time series** is data that changes over time — think daily temperature, monthly sales, or stock prices.

In [ ]:
# pd.date_range creates a list of dates — here every day from Jan 1 to Jun 30, 2021
dates = pd.date_range(start='2021-01-01', end='2021-06-30')
# Print how many dates were made
print(len(dates))

In [ ]:
# Build a small time-series DataFrame — dates + random temperatures
date_df = pd.DataFrame({'Date': dates})
# Add a random 'Temperature' column between 0 and 49 for every date
date_df['Temperature'] = np.random.randint(0, 50, size=len(dates))
date_df.head()

In [ ]:
# Plot the temperatures over time
date_df.plot.line(x='Date', y='Temperature', figsize=(10, 4), title='Daily Temperatures')
plt.show()

### Real time-series: Google stock prices

We'll load real Google stock data from a URL — thousands of rows spanning almost 20 years.

In [ ]:
# URL for Google stock prices (2004 → 2022) hosted on GitHub
url = "https://raw.githubusercontent.com/duochen/data-science-bootcamp/refs/heads/main/Data/google_stock_prices.csv"
# Read the CSV directly from the internet
google_stock = pd.read_csv(url)
# Convert the 'Date' column from plain text into real datetime objects
google_stock['Date'] = pd.to_datetime(google_stock['Date'])
# Use 'Date' as the DataFrame's index — required for time-series methods
google_stock.set_index('Date', inplace=True)
google_stock.head()

In [ ]:
# Plot the daily 'Open' price over all the years
google_stock['Open'].plot(figsize=(10, 4), title='Google Opening Price')
plt.show()

In [ ]:
# RESAMPLING = change the time granularity
# 'YE' = Year-End; .mean() gives the average price PER YEAR
yearly_open = google_stock['Open'].resample('YE').mean()
yearly_open

In [ ]:
# A bar chart makes the yearly trend easy to compare
yearly_open.plot(kind='bar', figsize=(10, 4), title='Google Yearly Average Open Price')
plt.show()

In [ ]:
# SHIFT moves rows up or down
# Shifting by 1 gives us the PREVIOUS day's price on each row — useful for comparisons
google_stock['Prev Open'] = google_stock['Open'].shift(1)
google_stock[['Open', 'Prev Open']].head()

In [ ]:
# ROLLING WINDOW = compute stats over a sliding window of rows
# Here: the average of the last 30 days at each point — a '30-day moving average'
google_stock['30-Day Avg'] = google_stock['Open'].rolling(window=30).mean()
# Plot the real price and the smoothed moving average side by side
google_stock[['Open', '30-Day Avg']].plot(figsize=(10, 4), title='Open Price vs 30-Day Moving Average')
plt.show()

## Great job!

You now know the advanced Pandas skills that show up in almost every real data-science project:

- **groupby / agg** — answer per-group questions
- **concat / merge** — combine multiple tables
- **drop_duplicates / pd.cut** — clean & categorize data
- **pivot_table / crosstab** — reshape data
- Built-in **plots**: histogram, line, scatter, bar, box, pie, KDE
- **Time-series**: `date_range`, `resample`, `shift`, and rolling windows

These tools are what turn raw data into insights — the whole point of data science.